# 02 - Data Validation & Quality Assessment

## Objective

Before building any Marketing Mix Model, validate that the incoming data is complete, consistent, and suitable for modelling.

This notebook performs:
- Schema validation
- Data type checks
- Duplicate detection
- Missing value analysis
- Outlier detection
- Range validation
- Correlation overview
- Data quality report


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd()
DATA_PATH = ROOT / "data" / "raw" / "marketing_mix_data.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["Week"])

print(df.shape)
df.head()


## 1. Schema Validation

In [ ]:

required_columns = [
    "Week","Sales","Revenue","Orders","Google_Search",
    "Meta","TV","Discount","Price","Competitor_Spend","Holiday"
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    print("Missing columns:", missing)
else:
    print("Schema validation passed.")


## 2. Data Types

In [ ]:

dtype_report = pd.DataFrame({
    "Column": df.columns,
    "Datatype": df.dtypes.astype(str)
})

display(dtype_report)


## 3. Missing Values

In [ ]:

missing = df.isna().sum().sort_values(ascending=False)

display(missing.to_frame("Missing Values"))

missing.plot.bar(figsize=(12,4), title="Missing Values")
plt.tight_layout()
plt.show()


## 4. Duplicate Records

In [ ]:

duplicates = df.duplicated().sum()
print("Duplicate Rows:", duplicates)


## 5. Descriptive Statistics

In [ ]:

display(df.describe().T)


## 6. Outlier Detection (IQR)

In [ ]:

numeric = df.select_dtypes(include=np.number)

summary=[]

for col in numeric.columns:
    q1=numeric[col].quantile(.25)
    q3=numeric[col].quantile(.75)
    iqr=q3-q1
    lower=q1-1.5*iqr
    upper=q3+1.5*iqr
    out=((numeric[col]<lower)|(numeric[col]>upper)).sum()
    summary.append([col,out])

outliers=pd.DataFrame(summary,columns=["Column","Outliers"])
display(outliers)


## 7. Correlation Matrix

In [ ]:

corr = numeric.corr()

plt.figure(figsize=(10,8))
plt.imshow(corr, aspect="auto")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.colorbar(label="Correlation")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


## 8. Data Quality Score

In [ ]:

checks = {
    "No Missing Values": df.isna().sum().sum()==0,
    "No Duplicates": duplicates==0,
    "Sales Positive": (df["Sales"]>0).all(),
    "Orders Positive": (df["Orders"]>0).all(),
    "Price Positive": (df["Price"]>0).all(),
}

quality = pd.DataFrame({
    "Check": checks.keys(),
    "Passed": checks.values()
})

display(quality)

score = quality["Passed"].mean()*100
print(f"Overall Data Quality Score: {score:.1f}%")


# Business Interpretation

A data scientist should never begin modelling before validating data quality.

Typical actions after this notebook:
- Fix schema issues
- Impute missing values
- Investigate outliers
- Validate business rules
- Document assumptions

## Interview Questions

1. Why validate data before modelling?
2. What is the difference between missing values and invalid values?
3. When should outliers be removed?
4. Why can duplicate records bias MMM?
5. Why is correlation inspection important before regression?
